In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:

# 1. Xác định thư mục data
ROOT = Path.cwd()
DATA_DIR = ROOT/ ".." / "data" / "raw"

# 2. Đọc dữ liệu CSV
CSV_PATH = DATA_DIR / "car_dataset.csv"

df = pd.read_csv(CSV_PATH)
print(df.head())

                       name  year  selling_price  km_driven    fuel  \
0             Maruti 800 AC  2007          60000      70000  Petrol   
1  Maruti Wagon R LXI Minor  2007         135000      50000  Petrol   
2      Hyundai Verna 1.6 SX  2012         600000     100000  Diesel   
3    Datsun RediGO T Option  2017         250000      46000  Petrol   
4     Honda Amaze VX i-DTEC  2014         450000     141000  Diesel   

  seller_type transmission         owner  
0  Individual       Manual   First Owner  
1  Individual       Manual   First Owner  
2  Individual       Manual   First Owner  
3  Individual       Manual   First Owner  
4  Individual       Manual  Second Owner  


In [4]:
# xóa dòng trùng lặp
df = df.drop_duplicates().reset_index(drop=True)
print(f"After deduplication: {df.shape[0]} rows")

After deduplication: 3577 rows


In [5]:
# 3.5 Feature Engineering

# Tạo feature tuổi xe
REFERENCE_YEAR = 2026
df['vehicle_age'] = REFERENCE_YEAR - df['year']
# log-transform target to reduce skew and improve linear model fit
df['log_selling_price'] = np.log1p(df['selling_price'])

# Hàm trích xuất hãng xe
def extract_brand(name):
    words = str(name).strip().split()

    # Xử lý riêng hãng Land Rover
    if len(words) >= 2 and words[0].lower() == 'land' and words[1].lower() == 'rover':
        return 'Land Rover'

    return words[0] if words else 'Unknown'

# Tạo cột brand
df['brand'] = df['name'].apply(extract_brand)
df.drop(columns = ['name', 'year', 'selling_price'], inplace=True)

print("Engineered features added: vehicle_age, brand")
df.head(3)

Engineered features added: vehicle_age, brand


,km_driven,fuel,seller_type,transmission,owner,vehicle_age,log_selling_price,brand
0,70000,Petrol,Individual,Manual,First Owner,19,11.002117,Maruti
1,50000,Petrol,Individual,Manual,First Owner,19,11.813037,Maruti
2,100000,Diesel,Individual,Manual,First Owner,14,13.304687,Hyundai


Feature Engineering – Brand Extraction

Cột name chứa thông tin về hãng, dòng xe và phiên bản xe, với 1.491 tên xe khác nhau.

Tiến hành trích xuất đặc trưng brand từ phần đầu của tên xe và xử lý trường hợp đặc biệt Land Rover.

Kết quả: Tạo được 29 nhóm tên hãng hoặc tên xe được trích xuất từ cột name.

Mục đích: Giảm số lượng giá trị phân loại cần mã hóa, đồng thời giữ lại thông tin về hãng xe để phục vụ dự đoán giá bán.

Đặc trưng brand sẽ được mã hóa trong giai đoạn Data Preprocessing trước khi huấn luyện mô hình.

In [6]:
# 3.6  Encode Categoricals 
# owner has ordinal meaning; map to integers
owner_map = {
    'First Owner'          : 1,
    'Second Owner'         : 2,
    'Third Owner'          : 3,
    'Fourth & Above Owner' : 4,
    'Test Drive Car'       : 0,
}
df['owner'] = df['owner'].map(owner_map)
# nominal categoricals → one-hot (drop_first avoids multicollinearity)
df = pd.get_dummies(
    df,
    columns=['brand', 'fuel', 'seller_type', 'transmission', ],
    drop_first=True,
    dtype=int
)
bool_cols = df.select_dtypes(bool).columns
df[bool_cols] = df[bool_cols].astype(int)
print(f"Final feature matrix: {df.shape}")
df.head(3)

Final feature matrix: (3577, 39)


,km_driven,owner,vehicle_age,log_selling_price,brand_Audi,brand_BMW,brand_Chevrolet,brand_Daewoo,brand_Datsun,brand_Fiat,...,brand_Toyota,brand_Volkswagen,brand_Volvo,fuel_Diesel,fuel_Electric,fuel_LPG,fuel_Petrol,seller_type_Individual,seller_type_Trustmark Dealer,transmission_Manual
0,70000,1,19,11.002117,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0,1
1,50000,1,19,11.813037,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0,1
2,100000,1,14,13.304687,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,1


Chuyển đổi các biến phân loại (Categorical) thành dạng số để sử dụng trong các mô hình Machine Learning.

1. Ordinal Encoding – owner

Mã hóa số đời chủ sở hữu theo thứ tự từ 1 đến 4.

Test Drive Car được gán giá trị 0 như một quy ước cho nhóm xe lái thử, không phản ánh thứ tự sở hữu thực tế.

2. One-Hot Encoding – brand, fuel, seller_type, transmission

Chuyển các biến phân loại không có thứ tự thành các cột nhị phân (0/1).

Sử dụng drop_first=True để loại bỏ một nhóm tham chiếu ở mỗi biến, giúp tránh đa cộng tuyến hoàn hảo giữa các biến giả khi mô hình có hệ số chặn.

Sử dụng dtype=int để tạo các cột nhị phân dạng số nguyên.

3. Chuẩn hóa kiểu dữ liệu

Chuyển các cột Boolean còn lại thành số nguyên (True → 1, False → 0) nếu có.

Kết quả: Các biến Categorical đã được mã hóa thành dạng số, sẵn sàng cho các bước tiếp theo trong quy trình xây dựng mô hình Machine Learning.

In [7]:
# 1. Xác định đường dẫn
ROOT = Path.cwd()
PROCESSED_DIR = ROOT / ".." / "data" / "processed"

# 2. Tạo thư mục nếu chưa tồn tại
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 3. Đường dẫn file CSV đã xử lý
CSV_PATH = PROCESSED_DIR / "car_processed.csv"

# 4. Lưu DataFrame thành CSV
df.to_csv(CSV_PATH, index=False)

print("Đã lưu file tại:", CSV_PATH.resolve())

Đã lưu file tại: D:\FPT\Semester 4\AIL303m\project\Salary Prediction\data\processed\car_processed.csv


In [ ]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "car_price_xgboost.joblib"

joblib.dump(best_xgb, MODEL_PATH)

print(f"[SAVED] {MODEL_PATH.resolve()}")
print(f"Features: {best_xgb.n_features_in_}")
